In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

### Brownian dynamics 

$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$

with potential

$$V(x) = (x_1^2-1)^2 + 2.0 * (x_1^2+x_2-1)^2$$ 

first, define $V$ and its gradient

In [ ]:
# a potential function
def V(X):
    return (X[0]**2 - 1)**2 + 2.0 * (X[0]**2 + X[1] - 1)**2

# gradient of potential function 
def gradV(X):
    return np.array(( 4.0 * X[0] * (X[0]**2 - 1.0 + 2.0*(X[0]**2 + X[1] - 1)), 4.0 * (X[0]**2 + X[1] - 1)) )

show the potential profile

In [ ]:
x = np.arange(-2.5, 2.5, 0.05)
y = np.arange(-2.5, 2.5, 0.05)
X, Y = np.meshgrid(x, y)

plt.figure(figsize = (6, 4))

contour_levels = [0.0, 1.0, 1.5, 2.0, 3.0, 4.0]

# evaluate potential on mesh
V_on_grid = V([X,Y])

fig = plt.figure(figsize=(7,4))
ax = fig.add_subplot(1, 1, 1)

# plot profile by pcolormesh
im = ax.pcolormesh(X, Y, V_on_grid, cmap='coolwarm',shading='auto', vmin=0, vmax=4)

# show contour lines
contours = ax.contour(X, Y, V_on_grid, contour_levels)
ax.clabel(contours, inline=True, fontsize=13,colors='black')

ax.set_aspect('equal')
ax.set_xlabel(r'$x_1$',fontsize=20)
ax.set_ylabel(r'$x_2$',fontsize=20, rotation=0)
ax.tick_params(axis='both', labelsize=20)

ax.set_xticks([-2.0, -1.0, 0, 1.0, 2.0])
ax.set_yticks([-2.0, -1.0, 0, 1.0, 2.0])
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-2.5, 2.5])

ax.set_title('V',fontsize=25)

# show colorbar
cbar = fig.colorbar(im, ax=ax, shrink=1.0)
cbar.ax.tick_params(labelsize=15)
plt.show()

### Compute mean by sampling SDE


We estimate the mean of a function $g$:

$$ \mathbb{E}_\pi(g) = \int_{\mathbb{R}^2} g(x) \pi(x) dx = \frac{1}{Z} \int_{\mathbb{R}^2} g(x) \mathrm{e}^{-\beta V(x)} dx,$$
where $\pi$ is the invariant density:

$$ \pi(x) = \frac{1}{Z} \mathrm{e}^{-\beta V(x)}.$$

### Method 1: 

By ergodic theorem, we have, with probability one (i.e. for almost all trajectories)

$$
 \lim_{T\rightarrow +\infty} \frac{1}{T} \int_0^T g(X_t) dt = \mathbb{E}_\pi(g)\,.
$$
where $X_t$ satisfies 
$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t.$$

Therefore, we can estimate the mean value of $g$ by computing the time average of $g$ along a **single, long** trajectory of the process $X_t$.

In practice, we approximate the time average on the left hand side above using discrete time points with finite time interval $[0,T]$: 
$$
\lim_{T\rightarrow +\infty} \frac{1}{T} \int_0^T g(X_t) dt \approx \frac{1}{T} \int_0^T g(X_t) dt \approx \frac{1}{N} \sum_{n=1}^N g(X_n)
$$
where $X_n$ are states of the process sampled at time $t_n = nh$, and $h=\frac{T}{N}$.

We take $g(x_1,x_2)=x_1$ and estimate the mean using:

$$\mathbb{E}_\pi(g) = \frac{1}{T} \int_0^T g(X_t) dt \approx \frac{1}{N} \sum_{n=1}^N g(X_n)$$




In [ ]:
def g(X):
    return X[0]

# sample the SDE using Euler-Maruyama scheme
def sample(X0, rng, beta=1.0, N=10000):
    X = X0
    traj = [X]
    delta_t = 0.001
    g_sum = 0.0
    for i in range(N):
        b = rng.normal(size=(2,))
        X = X - gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        g_sum = g_sum + g(X) 
        if i % 10 ==0:
            traj.append(X)

    return np.array(traj), g_sum / N

seed_list = [1, 199, 235, 37, 42]
X0 = [-1, 0]

# for each seed, generate a long trajectory 
for seed in seed_list:
    rng = np.random.default_rng(seed=seed)
    trajectory, g_mean = sample(X0, rng, beta=0.7, N=2000000)
    print (r'seed=%d, mean of g: %.4f' % (seed, g_mean))

### Method 2:  
  Use the fact that  $ p(x,t) \rightarrow \pi(x)$ as $t\rightarrow +\infty$.

  Therefore 
  $$ \mathbb{E}_\pi(g) = \frac{1}{Z} \int_{\mathbb{R}^2} g(x) \mathrm{e}^{-\beta V(x)} dx = \lim_{t\rightarrow + \infty} \int_{\mathbb{R}^2} g(x) p(x,t) dx$$

In practice, we approximate $p(x,t)$ using particles. Namely, $$p(x,t) \approx \frac{1}{M} \sum_{i=1}^{M} \delta(x-X^{(i)}_t)$$

#### Algorithm:

1. generate $M$ initial states $X_0^{(1)}, X_0^{(2)}, \dots, X_0^{(M)}$.
2. for each initial state $X_0^{(i)}$, sample the SDE $$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t, \quad X_0=X_0^{(i)}$$ up to time $t=T$. Let the final state to be $X_T^{(i)}$.
3. estimate
   $$\mathbb{E}_\pi(g) = \lim_{t\rightarrow + \infty} \int_{\mathbb{R}^2} g(x) p(x,t) dx \approx \frac{1}{M} \sum_{i=1}^M g(X_T^{(i)}).$$

set $\beta = 10.0$ and try:
1. $c_0=[-1.0,0]$
2. $c_0=[0, 0]$

In [ ]:
# vectorized version 
def gradV(X):
    return np.stack(( 4.0 * X[:,0] * (X[:,0]**2 - 1.0 + 2.0*(X[:,0]**2 + X[:,1] - 1)), 4.0 * (X[:,0]**2 + X[:,1] - 1)), axis=1)
    
def g1(X):
    return X[:,0]

# sample the SDE starting for multiple initial states. 
def sample(x0, rng, beta=1.0, N=10000):

    X = x0
    n_traj = x0.shape[0]   
    delta_t = 0.001
    for i in range(N):
        b = rng.normal(size=(n_traj, 2))
        X = X - gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b

    return X

# number of particles
n_traj = 1000
N = 100000
c0 = np.array([-1,0])
beta = 1.0

seed_list = [1, 199, 235, 37, 42]
# for each seed  
for seed in seed_list:
    rng = np.random.default_rng(seed=seed)
    x0 = c0 + 0.1 * rng.normal(size=(n_traj, 2)) 
    X = sample(x0, rng, beta=beta, N=N)
    mean_g = np.mean(g1(X))
    print (f'seed={seed}, mean of g: {mean_g}')

### Density evolution 

Brownian dynamics:
$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$

with a double-well potential function $V(x)=\frac{1}{4} (x^2-1)^2$.

Illustrate how the empirical density evolves as a function of $t$.

Compare the result for:

1. $\beta=2.0$ and $\beta=10.0$
2. $c_0=0.0$ and $c_0=-1.0$


In [ ]:
from scipy.stats import gaussian_kde

def gradV(x):
    return x*(x**2 - 1)

# sample the SDE starting for multiple initial states. 
def sample(x0, rng, save_N_list, beta=1.0, kappa=1.0, N=10000):
    X = x0
    n_traj = x0.shape[0]   
    delta_t = 0.001
    idx=0
    traj = []
    for i in range(N):
        if idx < len(save_N_list) and i == save_N_list[idx]:
            traj.append(X)
            idx += 1
        b = rng.normal(size=(n_traj, ))
        X = X - gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
    return np.array(traj)

beta = 2.0
c0 = -1.0
n_traj = 10000

save_N_list = [0, 1000, 5000, 10000, 20000]
times = [n * 1e-3 for n in save_N_list]

rng = np.random.default_rng(seed=100)

x0 = c0 + 0.5 * rng.normal(size=(n_traj,)) 

traj = sample(x0, rng, save_N_list=save_N_list, beta=beta, N=N)

fig, ax = plt.subplots(figsize=(10, 6))

lc = ['k', 'r', 'b', 'g', 'y']
for i, t in enumerate(times):
    kde = gaussian_kde(traj[i])
    x_range = np.linspace(-3,3, 200)
    density = kde(x_range)    
    ax.plot(x_range, density, color=lc[i], lw=1.5, label=f't={t}')

ax.legend()
ax.set_xlabel('x')
ax.set_ylabel('pdf')
ax.grid(axis='x', linestyle='--', alpha=0.6)

plt.show()